In [2]:
from typing import TypedDict,Literal 
from langchain_core.messages import HumanMessage,SystemMessage
from langchain_ollama import ChatOllama
from pydantic import BaseModel , Field
from dotenv import load_dotenv
import os 
import requests
from langchain_mcp_adapters.client import MultiServerMCPClient
import asyncio
import json
from swiggy_auth import create_oauth_provider
load_dotenv()

api_key = os.getenv("USDA_API_KEY")

class AgentState(TypedDict):
    user_query : str
    budget_amount: float | None
    protein_target: float | None
    protein_consumed: float | None
    protein_deficit: float | None
    recommendation : str
    approval:bool
    search_results: list[dict]
    ranked_items: list[dict]
    follow_up_question: str | None
   
llm = ChatOllama(model = "qwen2.5:1.5b")



In [3]:
class extract(BaseModel):
    amount : float|None  = Field(description ="extract the amount from the user input default value is none")
    protein_target : float|None = Field(description = "extract the target protein from the user input default value is none")
    protein_consumed : float|None = Field(description = "extract the consumed protein from the user query default value is none")

system_prompt = """
You are a constraint extraction AI.

Extract the following fields from the user's query:

1. amount
   - Budget amount mentioned by the user.

2. protein_target
   - Desired protein intake in grams.

3. protein_consumed
   - Protein already consumed by the user in grams.

Rules:
- Extract only information explicitly stated by the user.
- Never infer, estimate, or guess values.
- If a field is not mentioned, return None.
- If the query contains no relevant information, return None for all fields.
- Do not perform calculations.
- Do not assume that every number refers to every field.

Examples:

User: "I have ₹300"
amount = 300
protein_target = None
protein_consumed = None

User: "Need 120g protein"
amount = None
protein_target = 120
protein_consumed = None

User: "I've already consumed 40g protein"
amount = None
protein_target = None
protein_consumed = 40

User: "Recommend food"
amount = None
protein_target = None
protein_consumed = None
"""

llm_with_structure = llm.with_structured_output(extract)


def constraint_extract(state:AgentState):
    query = state["user_query"]
    response : extract = llm_with_structure.invoke([SystemMessage(content = system_prompt) , HumanMessage(content = query)])
    
    return {"budget_amount" : response.amount,
            "protein_target" : response.protein_target,
            "protein_consumed" : response.protein_consumed}


In [4]:

class followup(BaseModel):

    question : str

system_prompt2 = """
You are a follow-up question generation assistant.

Your task is to generate a natural question asking the user only for the missing constraints.

Rules:
- Ask only for the constraints provided.
- Do not ask for constraints that are already available.
- If multiple constraints are missing, combine them into a single concise question.
- Keep the question short and natural.
- Do not explain your reasoning.
- Do not mention constraint names literally unless they are user-friendly.
- Return only the question.
"""

def Missing_constraint(state:AgentState):
    missing = []

    if state["budget_amount"] is None:
        missing.append("budget")

    if state["protein_target"] is None:
        missing.append("protein_target")
    if not missing:
        return {
            "missing_constraints": [],
            "follow_up_question": None
        }
    
    llm_with_structure2 = llm.with_structured_output(followup)
    
    response: followup = llm_with_structure2.invoke(
        [
            SystemMessage(content=system_prompt2),
               HumanMessage(
    content=f"""
    The user has not provided the following information:

    {', '.join(missing)}

    Generate a follow-up question asking only for this information.
"""
)
            ])
        

    return {
        "follow_up_question": response.question
    }


In [5]:
oauth_provider = create_oauth_provider("https://mcp.swiggy.com/food")

client = MultiServerMCPClient({
    "swiggy-food": {
        "url": "https://mcp.swiggy.com/food",
        "transport": "streamable_http",
        "auth": oauth_provider,
    },
})

tools = await client.get_tools()

In [6]:
tools

[StructuredTool(name='get_addresses', description='Swiggy (Instamart/Food): Get all saved delivery addresses for the authenticated Swiggy user, sorted by last order date. This tool works for Swiggy Instamart and Food services. Addresses are returned WITHOUT coordinates (latitude/longitude) for privacy protection. No parameters needed - authentication is handled automatically.\n\n📍 IMPORTANT — STOP here and let the user choose:\n1. Show the address list to the user\n2. Ask: "Which address would you like to use for delivery?"\n3. Do NOT call any other tool until the user has selected an address\n4. Remember the selected addressId for all subsequent operations\n5. If no addresses are returned, inform the user that they need to add an address first', args_schema={'type': 'object', 'properties': {}, 'required': []}, metadata={'title': 'Get Delivery Addresses', 'readOnlyHint': True, 'destructiveHint': False, 'idempotentHint': True, 'openWorldHint': False, '_meta': {'ui': {'resourceUri': 'ui:

In [7]:
get_addresses = next(t for t in tools if t.name == "get_addresses")
get_restaurant = next(t for t in tools if t.name == "search_restaurants")
get_restaurant_menu = next(t for t in tools if t.name == "get_restaurant_menu")
search_menu = next(t for t in tools if t.name == "search_menu" )

In [ ]:
llm_search = ChatOllama(model = "qwen2.5:7b")
class search_query(BaseModel):
    search_query:str = Field(description= "store only the generated query from the llm default None")


system_prompt3 = """"You are a search query generator for an Indian food delivery platform.

Your task is to convert the user's food goal, nutritional goal, dietary preferences, and budget constraints into food search keywords that can be used in a food delivery application's search engine.

Generate 3-5 broad food search keywords, not specific menu items.
Prefer ingredients or food categories that are commonly used in food delivery searches.
Input may contain:

* User request
* Protein target
* Protein deficit
* Budget
* Vegetarian or non-vegetarian preference

Rules:

1. Return only food or dish names.
2. Return only English text.
3. Return 1 to 5 search terms.
4. Return a comma-separated list.
5. Do not explain your reasoning.
6. Do not return complete sentences.
7. Do not return JSON.
8. Do not return markdown.
9. Do not return restaurant names.
10. Do not return cuisine names such as:

* Chinese
* Indian
* Italian
* Mexican
* American

11. Do not return dietary labels such as:

* healthy
* vegan
* keto
* gluten-free
* low calorie
* high protein

12. Every output must be a real food item or dish that can be searched on a food delivery platform.
13. Prefer food items commonly available on Indian food delivery platforms.
14. Consider the user's nutritional goal when generating search terms.
15. Consider dietary restrictions when available.
16. Consider budget constraints when available.
17. Prioritize food items that help satisfy the remaining protein deficit when a protein deficit is provided.
18. If vegetarian preference is provided, generate only vegetarian food items.
19. If non-vegetarian preference is provided, prioritize chicken, eggs, fish, and other protein-rich non-vegetarian foods when appropriate.
20. Never output food items in Chinese, Japanese, Korean, or any non-English language.
21. Never output generic categories. Output only searchable dishes.

Good outputs:
chicken roll, grilled chicken, egg wrap
paneer tikka, paneer roll, soy chaap
egg sandwich, omelette, chicken sandwich
chicken biryani, grilled chicken, egg fried rice

Bad outputs:
Chinese, Italian, American
healthy, vegan, keto
KFC, McDonald's
鸡肉 stir-fry
30g protein under 300

Return only the search keywords.

"""""

llm_with_structure3 = llm_search.with_structured_output(search_query)


async def food_search(state:AgentState):
    budget = state["budget_amount"]
    query = state["user_query"]
    addresses_response = await get_addresses.ainvoke({})

    addresses = json.loads(addresses_response[0]["text"])
    question = "Which address would you like to use?"
    for i , address in enumerate(addresses , start = 1):
          question+=f"\n{i}. {address['label']}"
    address_id = None
    for address in addresses:
        if address["label"].lower() == query.lower():
            address_id = address["id"]
            break
       
    if address_id is None:
        return {
        "follow_up_question": question
        }   
    restaurant_query : search_query = llm_with_structure3.invoke([SystemMessage(content = system_prompt3), HumanMessage(content = query)])

    restaurants = await get_restaurant.ainvoke({"query": restaurant_query, "addressId" : address_id})
    
    for i , restaurants in enumerate(restaurants , start = 1):
        res_id = restaurants["restaurant_id"]
    menu  = search_menu.ainvoke({
        "restaurant_id" : res_id
    })


In [19]:
get_restaurant

StructuredTool(name='search_restaurants', description='Search and order food from restaurants for delivery. PRIMARY FOOD DELIVERY SERVICE - Use this when user wants to order food, get food delivered, or search restaurants for delivery. Swiggy Food delivery service. NOT for restaurant reservations or dine-out.\n\n⚠️ REQUIRED WORKFLOW: You MUST call get_addresses first to obtain a valid addressId, then pass that addressId to this tool. NEVER guess, invent, or use placeholder values like "default", "1", or "N/A". The addressId must come from get_addresses response.\n\nIMPORTANT: Each restaurant in the response includes an "availabilityStatus" field with values "OPEN", "CLOSED", or "UNAVAILABLE". Always check this status before proceeding: only recommend or add items from restaurants with availabilityStatus "OPEN". If a restaurant is "CLOSED" or "UNAVAILABLE", inform the user and suggest open alternatives from the results.\n\nAfter showing results, let the user pick a restaurant before sea

In [58]:
restaurants = await get_restaurant.ainvoke({"query": "chicken",
    "addressId": "d7dj5lfj7k7gunt915h0"})
restaurants

[{'type': 'text',
  'text': 'Found 10 restaurants for "chicken":\n1. Hot Chicken Wings - 4 pcs —  | undefined★ | ? min |  (ID: 63663293)\n2. Chicken Biriyani —  | undefined★ | ? min |  (ID: 182238861)\n3. Chicken Doubles Pizza Party —  | undefined★ | ? min |  (ID: 193126719)\n4. Chicken Chaap —  | undefined★ | ? min |  (ID: 184055892)\n5. Chicken Moburg —  | undefined★ | ? min |  (ID: 95598399)\n6. Ultimate Savings Chicken Bucket —  | undefined★ | ? min |  (ID: 145480325)\n7. Chicken Moburg —  | undefined★ | ? min |  (ID: 95598906)\n8. Chicken Biryani —  | undefined★ | ? min |  (ID: 200347673)\n9. Chicken Kosha —  | undefined★ | ? min |  (ID: 194949424)\n10. Boneless Chilli Chicken —  | undefined★ | ? min |  (ID: 38540250)\n\n⚠️ A rich UI widget is being shown to the user with this data. Do NOT repeat or list the same information in your response — the user can already see it. Instead, provide a brief recommendation or ask the user what they\'d like to do next.',
  'id': 'lc_92980070-3

In [55]:
from pprint import pprint

menu = await search_menu.ainvoke({
    "query": "chicken",
    "addressId": "d7dj5lfj7k7gunt915h0"
})

print(type(menu))
pprint(menu)

<class 'list'>
[{'id': 'lc_97a906fc-4fff-4dd9-97da-4bdf1ae6932f',
  'text': 'Found 0 menu items for "chicken":\n'
          '\n'
          '\n'
          '⚠️ A rich UI widget is being shown to the user with this data. Do '
          'NOT repeat or list the same information in your response — the user '
          'can already see it. Instead, provide a brief recommendation or ask '
          "the user what they'd like to do next.",
  'type': 'text'}]
